# 02 — Data Cleaning & Preprocessing

Goal: turn the raw dataset into a clean, fully-numeric dataset that's ready for model training, and save it to `data/processed/`.

## 1. Imports & load raw data

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.shape

## 2. Drop customerID

It's a unique identifier with no predictive value — keeping it risks the model treating it as a feature.

In [ ]:
df = df.drop(columns=["customerID"])

## 3. Fix TotalCharges dtype

Convert to numeric. The blank-string rows (found on Day 2) will become `NaN` after this conversion —
that's expected, we handle them in the next step.

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"].isnull().sum()

## 4. Handle the blank TotalCharges rows

These are all customers with `tenure == 0` — brand new customers who haven't been billed yet.
Two reasonable options:
- **Drop them** (only 11 rows, ~0.16% of data — negligible loss, and they carry no churn history since they just joined)
- **Fill with 0** (logically consistent: 0 tenure = 0 total charges so far)

We'll drop them — it's simpler and the sample size loss is negligible. Document this choice in your notes,
since a reviewer may ask about it.

In [ ]:
df = df.dropna(subset=["TotalCharges"]).reset_index(drop=True)
df.shape

## 5. Collapse "No service" categories

Columns like `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`
contain `"No internet service"` as a value, and `MultipleLines` contains `"No phone service"`.
These are functionally the same as `"No"` — collapse them so the encoding step doesn't create
redundant extra categories.

In [ ]:
no_internet_cols = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
                     "TechSupport", "StreamingTV", "StreamingMovies"]
for col in no_internet_cols:
    df[col] = df[col].replace("No internet service", "No")

df["MultipleLines"] = df["MultipleLines"].replace("No phone service", "No")

## 6. Encode the target variable

Convert `Churn` from Yes/No text to 1/0 — required for every scikit-learn classifier.

In [ ]:
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

## 7. Encode binary categorical columns

Columns with exactly two categories can be safely label-encoded (0/1) without implying an order.

In [ ]:
binary_cols = ["gender", "Partner", "Dependents", "PhoneService",
               "PaperlessBilling", "MultipleLines"] + no_internet_cols
# print unique values to double check before mapping
for col in binary_cols:
    print(col, df[col].unique())

In [ ]:
binary_map = {"Yes": 1, "No": 0, "Male": 1, "Female": 0}
for col in binary_cols:
    df[col] = df[col].map(binary_map)

# SeniorCitizen is already 0/1 — no change needed
df[binary_cols + ["SeniorCitizen"]].head()

## 8. One-hot encode multi-category columns

Columns with 3+ categories (`InternetService`, `Contract`, `PaymentMethod`) get one-hot encoded —
label encoding these would wrongly imply an order (e.g. that "Two year" > "Month-to-month" numerically).

In [ ]:
multi_cat_cols = ["InternetService", "Contract", "PaymentMethod"]
df = pd.get_dummies(df, columns=multi_cat_cols, drop_first=True)
df.head()

## 9. Final sanity check

Every column should now be numeric, and there should be no missing values left.

In [ ]:
df.info()

In [ ]:
df.isnull().sum().sum()  # should be 0

## 10. Save the cleaned dataset

This is the Day 3 deliverable — `data/processed/telco_churn_clean.csv`, ready to be loaded
directly for model training on Day 4.

In [ ]:
df.to_csv("../data/processed/telco_churn_clean.csv", index=False)
print("Saved:", df.shape)

## 11. Notes on scaling

We did **not** scale numeric columns (`tenure`, `MonthlyCharges`, `TotalCharges`) here, on purpose.
Scaling should happen *after* the train/test split (Day 4) to avoid **data leakage** — if you scale
before splitting, information about the test set's distribution leaks into training. Tree-based models
(Random Forest, XGBoost) don't need scaling at all; only distance-based models (Logistic Regression, KNN, SVM) do.

## Cleaning Summary

*(Replace with your own actual results)*

- Dropped `customerID` (identifier, not a feature)
- Converted `TotalCharges` to numeric; dropped 11 rows with 0 tenure and blank charges
- Collapsed "No internet/phone service" into "No" across 7 columns
- Encoded `Churn` target as 1/0
- Label-encoded 8 binary columns, one-hot encoded 3 multi-category columns
- Final shape: (rows, columns) — fill in from output above
- Saved to `data/processed/telco_churn_clean.csv`
